# buffer — Hugging Face model smoke test

Load and run a short generation for Qwen, Llama, and OLMo 3.

**Prerequisites**
- `uv sync` (includes torch, transformers, accelerate)
- GPU recommended (~16GB+ VRAM for 7–8B models)
- Llama is gated: `huggingface-cli login` and request access first

Run one model at a time if VRAM is tight — each test frees memory before the next.

In [1]:
import gc
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/Users/calebtalley/Projects/USC/DILL/StutterMark/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
MODELS = {
    "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
    "llama": "meta-llama/Llama-3.2-1B-Instruct",  # gated — needs HF login
    "olmo": "allenai/OLMo-2-0425-1B-Instruct",  # OLMo 2; OLMo 3 starts at 7B
}

PROMPT = "What is 2+2? Answer briefly."

In [3]:
def smoke_test(name: str, model_id: str, prompt: str = PROMPT, max_new_tokens: int = 32):
    """Load a model, generate once, print timing, then free memory."""
    print(f"\n{'=' * 60}")
    print(f"{name}: {model_id}")
    print('=' * 60)

    t0 = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )
    load_s = time.perf_counter() - t0
    print(f"Loaded in {load_s:.1f}s")

    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t1 = time.perf_counter()
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_s = time.perf_counter() - t1

    new_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"Generation: {gen_s:.2f}s ({new_tokens} tokens, {gen_s / new_tokens * 1000:.0f} ms/token)")
    print(f"Response: {response!r}")

    del model, tokenizer, inputs, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"load_s": load_s, "gen_s": gen_s, "new_tokens": new_tokens, "response": response}

In [4]:
results = {}
for name, model_id in MODELS.items():
    try:
        results[name] = smoke_test(name, model_id)
    except Exception as e:
        print(f"FAILED: {e}")
        results[name] = {"error": str(e)}


qwen: Qwen/Qwen2.5-0.5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 35335.33it/s]


Loaded in 71.8s
Generation: 3.64s (9 tokens, 405 ms/token)
Response: '2 + 2 equals 4.'

llama: meta-llama/Llama-3.2-1B-Instruct
FAILED: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
403 Client Error. (Request ID: Root=1-6a91ba1e-7d5198e67bdb9239161b6724;c2bb83bc-ad7b-4eb0-8c68-e374788fc0fb)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct to ask for access.

olmo: allenai/OLMo-2-0425-1B-Instruct
FAILED: Can't load the model for 'allenai/OLMo-2-0425-1B-Instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'allenai/OLMo-2-0425-1B-Instruct' is the correct path to a direct

In [ ]:
results